In [15]:
import pandas as pd
import numpy as np

day_1 = pd.read_sas("Data/DR1TOT_J.xpt")
day_2 = pd.read_sas("Data/DR2TOT_J.xpt")
demo = pd.read_sas("Data/DEMO_J.xpt")

### Przegląd danych

In [2]:
# print(day_1.head())
# print(day_2.head())
# print(demo.head())

# day_1.shape
# day_2.shape
# demo.shape

# day_1.info()
# day_2.info()
# demo.info()

# day_1.describe()
# day_2.describe()
demo.describe()

### Co oznaczają kolumny
Opis kolumn DR1TOT_J https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DR1TOT_J.htm#SEQN  

**SEQN** - id uzytkownika  
**WTDRD1** - Oznacza ile osób w populacji USA reprezentuje dany respondent. 5.397605346934028e-79  jest ich 1063 i to oznacza ze Day 1 dietary recall not done/incomplete.
**WTDRD2** - Oznacza ile osób w populacji USA reprezentuje dany respondent.  5.397605346934028e-79 jest ich 1002 i to oznacza ze Day 2 dietary recall not done/incomplete.
**DR1DRSTZ** - Okreslenie jak wiarygodne jest ich raportowania z dnia zywienia. Czemu jest 1,2 4,5 ? Gdzie 3 ? 4 oznacza diete oparta na mleku matki więc nalezalloby to usunąć w dalszych analizach.  
**DR1EXMER** - identyfikator osoby przeprowadzającej wywiad  
**DRABF** - Oznaczenie niemowląt które są na diecie mleka matki  
**DRDINT** - oznakowanie z ilu dni były zebrane dane 1 czy 2  
**DR1DBIH** - liczba dni miedzy dniem spozycia (dietary recall) a wywiadem. Ile dni minelo miedzy dniem którego dotyczy dieta a dniem przeprowadzonego wywiadu. Kolumna istnieje zeby okreslic ryzyko błędu pamięci ? Czyli im wieksza wartosc tym mniej wartosciowe dane  
**DR1DAY** - oznakowanie w jakim dniu jest wspomniana dieta 1-7 Ndz - Sob  
**DR1LANG** - język, którego uzywali respondenci. Tabela na stronie  
**DR1MRESP** - kto głównie odpowiadał na pytania do ankeitowanego poniewaz czesto były wywiady przeprowadzane w domu gdzie była obecna rodzina, lub np respondent nie był ws tanie odpowiedziec na pytanie (np. niemowlę lub bariera jezykowa).  Oznaczenie 1-3 oznacza raczej najbardziej wiarygodne. Tabela na stronie  
**DR1HELP** - kto pomogał przy odpowiedziach. Najczystszy przypadek danych jest 12- No one poniewaz respondent sam odpowiadał bez pomocy. Tabela na stronie  

### Oczyszczanie danych - określam dane jako goden zaufania, brak duplikatów, nulli czy niezgodności

In [3]:
# day_1.duplicated().sum() # 0
# day_2.duplicated().sum() # 0
demo.duplicated().sum() # 0

day_1.isnull().sum()

SEQN           0
WTDRD1         0
WTDR2D      1063
DR1DRSTZ       0
DR1EXMER     976
            ... 
DRD370T     4366
DRD370TQ    7662
DRD370U     4366
DRD370UQ    8454
DRD370V     4367
Length: 168, dtype: int64

### Buduje pipeline warstę SILVER

In [ ]:

silver_demo = demo[[
    "SEQN",
    "RIAGENDR",
    "RIDAGEYR"

]]

silver_day_1 = day_1[[
    "SEQN",
    "DR1DRSTZ",
    "DR1MRESP",
    "DR1HELP",
    "DBQ095Z",
    "DBD100",
    "DR1STY",
    "DRQSDIET",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TCARB",
    "DR1TSUGR",
    "DR1TFIBE",
    "DR1TTFAT",
    "DR1TSODI",
    "DR1TALCO"
]]

silver_df = silver_day_1.merge(silver_demo, on="SEQN", how="left")

# zastanowic sie ktore rzeczy usuwam i czy daje cos takiego 
# --- flags jakości ---
# silver["is_infant"] = silver["RIDAGEYR"] < 2

# silver["is_proxy"] = silver["DR1MRESP"] != 1

# silver["is_helped"] = silver["DR1HELP"] != 12

# # --- proste grupy wieku ---
# silver["age_group"] = pd.cut(
#     silver["RIDAGEYR"],
#     bins=[0, 18, 65, 120],
#     labels=["child", "adult", "senior"]
# )